### CArga de bases


In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()



In [2]:
fecha_mes_base='2026-08-01'
fecha_envio_base='2026-08-01'
filename='Base_PPD_TARGET_Agosto2026_ACT.xlsx'
ruta_archivo = os.path.join(ruta_csv, filename)
df_recepcion_ppd = pd.read_excel(ruta_archivo)

filename='Base_PPDPLUS_TARGET_Agosto2026_ACT.xlsx'
ruta_archivo = os.path.join(ruta_csv, filename)
df_recepcion_plus = pd.read_excel(ruta_archivo)

print(df_recepcion_ppd.shape,'ppd')
print(df_recepcion_plus.shape,'plus')
print(df_recepcion_ppd.columns.tolist())
print(df_recepcion_plus.columns.tolist())


(12742, 87) ppd
(1242, 88) plus
['Nro_Tarjeta', 'DNI', 'CUS1', 'IdenCuenta', 'Probabilidad', 'Nombre_Producto', 'TipoTarjeta', 'Estado_Tarjeta', 'TEA_FORMATO', 'TEM_FORMATO', 'MONTO_PPD', 'prioridad_PPD', 'Deuda_tc_BCP', 'Deuda_tc_Continental', 'Deuda_tc_ScotiaBank', 'Deuda_tc_Interbank', 'Deuda_tc_falabella', 'Deuda_tc_total', 'Fecha_Expiracion', 'Tel1', 'Tel2', 'Tel3', 'Tel4', 'Nombre_Largo', 'Nombre', 'ApPaterno', 'ApMaterno', 'Ciclo', 'FechaNacimiento', 'Nacionalidad', 'EstadoCivil', 'Conyuge', 'Alerta_Diners', 'Direccion_Correspondencia', 'Distrito_Correspondencia', 'Provincia_Correspondencia', 'Departamento_Correspondencia', 'Direccion_Domicilio', 'Distrito', 'Provincia', 'Departamento', 'RazonSocial_Empresa', 'RUC_Empresa', 'Cargo', 'Profesion', 'Direccion_Lab', 'Distrito_Lab', 'Provincia_Lab', 'Departamento_Lab', 'Flag_entrega', 'mes_entrega', 'FLAG_ADICION', 'FLAG_CAD', 'NOMBRE_adic1', 'ApPaterno1', 'ApMaterno1', 'NOMBRE_adic2', 'ApPaterno2', 'ApMaterno2', 'NOMBRE_adic3', 'ApP

In [3]:
df_recepcion_ppd['DNI'] = df_recepcion_ppd['DNI'].astype(str).str.zfill(8)

print('Total recibido:',df_recepcion_ppd.shape[0])
print('Total recibido sin duplicados:',df_recepcion_ppd['DNI'].nunique())

df_recepcion_plus['DNI'] = df_recepcion_plus['DNI'].astype(str).str.zfill(8)

print('Total recibido:',df_recepcion_plus.shape[0])
print('Total recibido sin duplicados:',df_recepcion_plus['DNI'].nunique())

Total recibido: 12742
Total recibido sin duplicados: 12742
Total recibido: 1242
Total recibido sin duplicados: 1242


In [4]:

ruta_archivo = os.path.join(ruta_csv, 'recepcion_mes_ssff_ppd.csv')
df_recepcion_ppd.to_csv(ruta_archivo,index=False,sep=';',encoding='utf-8-sig')
ruta_archivo = os.path.join(ruta_csv, 'recepcion_mes_ssff_plus.csv')
df_recepcion_plus.to_csv(ruta_archivo,index=False,sep=';',encoding='utf-8-sig')

In [5]:
fecha_envio_base


'2026-08-01'

In [6]:

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_kishin}:{pwd_kishin}@{server_kishin}/{db_kishin}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
    SELECT *
    FROM DANTALION.dbo.Base_Maestra_Diners
    WHERE fecha_envio >='{fecha_envio_base}'
    and TIPO_PRODUCTO in ('PPD','PPD PLUS')
"""
df_vigente = pd.read_sql(query, engine_kishin)

In [7]:
dni_vigente_ppd = set(
    df_vigente.loc[df_vigente["TIPO_PRODUCTO"] == "PPD", "NumDoc"]
    .dropna()
    .astype(str)
    .str.strip()
)
dni_vigente_plus = set(
    df_vigente.loc[df_vigente["TIPO_PRODUCTO"] == "PPD PLUS", "NumDoc"]
    .dropna()
    .astype(str)
    .str.strip()
)
dni_recepcion_ppd = set(df_recepcion_ppd['DNI'].dropna().astype(str))
dni_recepcion_plus = set(df_recepcion_plus['DNI'].dropna().astype(str))


In [8]:
dni_nuevos_ppd = dni_recepcion_ppd - dni_vigente_ppd
dni_nuevos_plus = dni_recepcion_plus - dni_vigente_plus
print("Lead nuevos ppd:", len(dni_nuevos_ppd))
print("Lead nuevos plus:", len(dni_nuevos_plus))

dni_comunes_ppd = dni_vigente_ppd & dni_recepcion_ppd
dni_comunes_plus = dni_vigente_plus & dni_recepcion_plus
print("Lead en comun ppd:", len(dni_comunes_ppd))
print("Lead en comun plus:", len(dni_comunes_plus))

dni_retirados_ppd = dni_vigente_ppd - dni_recepcion_ppd
dni_retirados_plus = dni_vigente_plus - dni_recepcion_plus
print("Lead retirados:", len(dni_retirados_ppd))
print("Lead recepcion:", len(dni_retirados_plus))

Lead nuevos ppd: 1214
Lead nuevos plus: 404
Lead en comun ppd: 11528
Lead en comun plus: 838
Lead retirados: 363
Lead recepcion: 99


In [13]:



dni_nuevos = dni_recepcion - dni_vigente
if dni_nuevos:
    df_nuevos = pd.DataFrame({
        'NUMERO_DOCUMENTO': sorted(dni_nuevos)
    })

    ruta_archivo = os.path.join(ruta_csv, 'recepcion_nuevo_tc.csv')
    df_nuevos.to_csv(ruta_archivo,index=False,encoding='utf-8-sig')
    print('se guardo registros nuevos')
else:
    print('no se tienen regisros nuevos')

dni_comunes = dni_vigente & dni_recepcion
if dni_comunes:
    df_nuevos = pd.DataFrame({
        'NUMERO_DOCUMENTO': sorted(dni_comunes)
    })

    ruta_archivo = os.path.join(ruta_csv, 'recepcion_comun_tc.csv')
    df_nuevos.to_csv(ruta_archivo,index=False,encoding='utf-8-sig')
    print('se guardo regisros en comun')
else:
    print('no se tienen regisros en comun')


NameError: name 'dni_recepcion' is not defined

In [ ]:
SI NO SE TIENE REGISTROS ENCOMUN PUES EL ARCHIVO COMUN DE ESTAR VACIO

In [13]:
query = f"""
    SELECT *
    FROM DANTALION.dbo.Base_Maestra_Diners_vigente
    where TIPO_PRODUCTO in ('PPD','PPD PLUS')
    """
df_formato_1=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)


In [ ]:
query = f"""
    SELECT *
    FROM DANTALION.dbo.Base_Maestra_Diners
    WHERE fecha_envio >='{fecha_envio_base}'
    and TIPO_PRODUCTO in ('PPD','PPD PLUS')
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

query = f"""
    SELECT *
    FROM DANTALION.dbo.Base_Maestra_Diners_vigente
    and TIPO_PRODUCTO in ('PPD','PPD PLUS')
    """
df_formato_1=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

filename='recepcion_mes_ssff_ppd.csv'
df_base_ppd=cargar_archivo_csv(spark,filename,';',True)

filename='recepcion_mes_ssff_plus.csv'
df_base_plus=cargar_archivo_csv(spark,filename,';',True)


# overwrite_table_SQL(spark,df_comun,f'borrar_base_actual_diners_tc',server_kishin,user_kishin,pwd_kishin,'DANTALION')

print(df_base_ppd.count())
print(df_base_plus.count())

12742
1242


In [10]:
df_base_ppd = df_base_ppd.withColumn(
        "DNI",
        F.lpad(F.col("DNI").cast("string"), 8, "0")
    )
df_base_plus = df_base_plus.withColumn(
        "DNI",
        F.lpad(F.col("DNI").cast("string"), 8, "0")
    )

In [11]:
exprs = [
    F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in df_base_ppd.columns
]

df_counts = df_base_ppd.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base_ppd = df_base_ppd.select(cols_con_data)

exprs = [
    F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in df_base_plus.columns
]

df_counts = df_base_plus.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base_plus = df_base_plus.select(cols_con_data)

In [17]:
df_base_ppd=df_base_ppd.withColumnRenamed('CANAL_ASIGNADO','LIDER_CARTERA')
# df_base_ppd=df_base_ppd.withColumnRenamed('Prioridad_Modelo_PPD_efec','MARCA2')
# df_base_ppd=df_base_ppd.withColumnRenamed('prioridad_PPD','prioridad_inicial')
df_base_ppd=df_base_ppd.withColumnRenamed('ASESOR_ASIGNADO','ASESOR_CARTER')
# df_base_ppd=df_base_ppd.withColumnRenamed('TIPO_PRODUCTO','MARCA')
df_base_ppd=df_base_ppd.withColumnRenamed('SALDO_PPD','saldo_ppd')
df_base_ppd=df_base_ppd.withColumnRenamed('MONTO_PPD','Linea_EI_60M')
df_base_ppd=df_base_ppd.withColumnRenamed('DNI','NumDoc')
df_base_ppd=df_base_ppd.withColumnRenamed('Deuda_tc_Continental','Deuda_Continental')
df_base_ppd=df_base_ppd.withColumnRenamed('Deuda_tc_Interbank','Deuda_Interbank')
df_base_ppd=df_base_ppd.withColumnRenamed('Deuda_tc_BCP','Deuda_BCP')
df_base_ppd=df_base_ppd.withColumnRenamed('Deuda_tc_falabella','Deuda_Otros')
df_base_ppd=df_base_ppd.withColumnRenamed('Deuda_tc_ScotiaBank','Deuda_ScotiaBank')
df_base_ppd=df_base_ppd.withColumnRenamed('RECURRENCIA_DIGITAL','MARCA3')
df_base_ppd=df_base_ppd.withColumnRenamed('TASA_CREDITO_ORIGINAL','TASA_MACRO')
df_base_ppd=df_base_ppd.withColumn('MPPD',F.lit('PPD'))
df_base_ppd=df_base_ppd.withColumn('MARCA',F.lit('PPD'))
df_base_ppd=df_base_ppd.withColumn('TIPO_PRODUCTO',F.lit('PPD'))

df_base_ppd=df_base_ppd.withColumnRenamed('TASA_CRED_ORIGINAL','TASA_MACRO')
df_base_ppd=df_base_ppd.withColumnRenamed('Score','prioridad_inicial')

In [18]:
df_base_plus=df_base_plus.withColumnRenamed('CANAL_ASIGNADO','LIDER_CARTERA')
# df_base_plus=df_base_plus.withColumnRenamed('Prioridad_Modelo_PPD_efec','MARCA2')
# df_base_plus=df_base_plus.withColumnRenamed('prioridad_PPD','prioridad_inicial')
df_base_plus=df_base_plus.withColumnRenamed('ASESOR_ASIGNADO','ASESOR_CARTER')
# df_base_plus=df_base_plus.withColumnRenamed('TIPO_PRODUCTO','MARCA')
df_base_plus=df_base_plus.withColumnRenamed('SALDO_PPD','saldo_ppd')
df_base_plus=df_base_plus.withColumnRenamed('MONTO_PPD','Linea_EI_60M')
df_base_plus=df_base_plus.withColumnRenamed('DNI','NumDoc')
df_base_plus=df_base_plus.withColumnRenamed('Deuda_tc_Continental','Deuda_Continental')
df_base_plus=df_base_plus.withColumnRenamed('Deuda_tc_Interbank','Deuda_Interbank')
df_base_plus=df_base_plus.withColumnRenamed('Deuda_tc_BCP','Deuda_BCP')
df_base_plus=df_base_plus.withColumnRenamed('Deuda_tc_falabella','Deuda_Otros')
df_base_plus=df_base_plus.withColumnRenamed('Deuda_tc_ScotiaBank','Deuda_ScotiaBank')
df_base_plus=df_base_plus.withColumnRenamed('RECURRENCIA_DIGITAL','MARCA3')
df_base_plus=df_base_plus.withColumnRenamed('TASA_CREDITO_ORIGINAL','TASA_MACRO')
df_base_plus=df_base_plus.withColumn('MPPD',F.lit('PPD PLUS'))
df_base_plus=df_base_plus.withColumn('MARCA',F.lit('PPD PLUS'))
df_base_plus=df_base_plus.withColumn('TIPO_PRODUCTO',F.lit('PPD PLUS'))

df_base_plus=df_base_plus.withColumnRenamed('TASA_CRED_ORIGINAL','TASA_MACRO')
df_base_plus=df_base_plus.withColumnRenamed('Score','prioridad_inicial')

In [19]:
df_base_ppd=df_base_ppd.drop('Deuda_tc_total','prioridad_PPD','SEXO', 'TASA_CRM', 'DIF_TASA', 'BASE')
df_base_plus=df_base_plus.drop('Deuda_tc_total','prioridad_PPD','SEXO', 'TASA_CRM', 'DIF_TASA', 'BASE')

In [20]:
df_base_ppd=df_base_ppd.withColumnRenamed('TASA_PREFERENCIAL','TASA_APP')
df_base_ppd=df_base_ppd.withColumnRenamed('CORREO','correo')
df_base_ppd=df_base_ppd.withColumnRenamed('FRECUENCIA_CAMPANA_PPD_U6M','PRIORIDAD_MODELO')

df_base_plus=df_base_plus.withColumnRenamed('TASA_PREFERENCIAL','TASA_APP')
df_base_plus=df_base_plus.withColumnRenamed('FRECUENCIA_CAMPANA_PPD_U6M','PRIORIDAD_MODELO')
df_base_plus=df_base_plus.withColumnRenamed('CORREO','correo')

In [22]:

df_base_ppd=df_base_ppd.withColumnRenamed('DNI_Asesor','ASESOR_CARTER')
df_base_plus=df_base_plus.withColumnRenamed('DNI_Asesor','ASESOR_CARTER')

In [23]:
cols_prueba = set(df_base_ppd.columns)
cols_prueba_ppdplus = set(df_base_plus.columns)
cols_formato = set(df_formato.columns)

solo_en_prueba_ppd = cols_prueba - cols_formato
print("Solo falta estas columnas en ppd:", solo_en_prueba_ppd)
solo_en_prueba_ppdplus = cols_prueba_ppdplus - cols_formato
print("Solo falta estas columnas en ppdplus:", solo_en_prueba_ppdplus)

Solo falta estas columnas en ppd: set()
Solo falta estas columnas en ppdplus: set()


In [24]:
df_base_ppd=df_base_ppd.withColumn('FECHA_ENVIO',F.lit('2026-08-14'))
df_base_ppd=df_base_ppd.withColumn('MES_DURACION_BASE',F.lit('08'))
df_base_ppd=df_base_ppd.withColumn('AÑO_DURACION_BASE',F.lit('2026'))
df_base_ppd=df_base_ppd.withColumn('SERVICIO',F.lit('01'))
df_base_plus=df_base_plus.withColumn('FECHA_ENVIO',F.lit('2026-08-14'))
df_base_plus=df_base_plus.withColumn('MES_DURACION_BASE',F.lit('08'))
df_base_plus=df_base_plus.withColumn('AÑO_DURACION_BASE',F.lit('2026'))
df_base_plus=df_base_plus.withColumn('SERVICIO',F.lit('01'))

In [25]:
df_base_ppd = df_base_ppd.withColumn(
    "CEL01",
    F.when(F.col("Tel1").rlike(r"^9\d{0,8}$"), F.col("Tel1"))
)
df_base_ppd = df_base_ppd.withColumn(
    "CEL02",
    F.when(F.col("Tel2").rlike(r"^9\d{0,8}$"), F.col("Tel2"))
)
df_base_ppd = df_base_ppd.withColumn(
    "CEL03",
    F.when(F.col("Tel3").rlike(r"^9\d{0,8}$"), F.col("Tel3"))
)


In [26]:
df_base_plus = df_base_plus.withColumn(
    "CEL01",
    F.when(F.col("Tel1").rlike(r"^9\d{0,8}$"), F.col("Tel1"))
)
df_base_plus = df_base_plus.withColumn(
    "CEL02",
    F.when(F.col("Tel2").rlike(r"^9\d{0,8}$"), F.col("Tel2"))
)

In [38]:
df_base_ppd=df_base_ppd.drop('ASESOR_CARTER')
df_base_plus=df_base_plus.drop('ASESOR_CARTER')

In [30]:
df_base_ppd.show(4)

+--------------+--------+----------+---------------+-----------+--------------+------------------+-----------+------------+---------+-----------------+----------------+---------------+-----------+----------------+---------+----+----+--------------------+--------+---------+---------+-----+---------------+------------+--------------------+-------------------------+------------------------+-------------------------+----------------------------+--------------------+-----------------+---------+------------+--------------------+-----------+-------------+--------------------+--------------------+-----------------+-------------+----------------+------------+-----------+------------------+------------+------------+----------+----------+------------+----------+----------+------------+----------+----------+------------+----------+----------+------------+----------+----------+---------+-----------------+--------+----------------+--------------------+------------+---------+-------------+---------+-

In [31]:
df_base_ppd_ctrl=df_base_ppd.select('NumDoc','TIPO_PRODUCTO')
df_base_plus_ctrl=df_base_plus.select('NumDoc','TIPO_PRODUCTO')

df_ctrl=df_base_ppd_ctrl.union(df_base_plus_ctrl)

In [30]:
df_formato_ctrl=df_formato.select('NumDoc',F.lit('en_base').alias('tipo_base'))

In [32]:
df_ctrl=df_ctrl.join(df_formato_ctrl,['NumDoc'],'left')

In [33]:
df_ctrl=df_ctrl.withColumn('tipo_base',when(F.col('tipo_base').isNull(),F.lit('nuevo'))
                                        .otherwise(F.col('tipo_base')))

In [ ]:

# append_table_SQL(spark,df_ctrl,f'borrar_ctrol_diners_ssff',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [36]:
print(df_base_ppd.count())
print(df_base_plus.count())

12742
1242


In [39]:
append_table_SQL(spark,df_base_ppd,f'Base_Maestra_Diners',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [32]:
print(df_base_ppd.count())
print(df_base_plus.count())

11891
937


In [40]:
append_table_SQL(spark,df_base_plus,f'Base_Maestra_Diners',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [41]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 46.82 seg
SP actualizar diners Zeus | realizado | duración: 136.71 seg
SP actualizar diners SA | realizado | duración: 2.8 seg


In [23]:
print([row['FRECUENCIA_CAMPANA_PPD_U6M' ] for row in df_base_ppd.select('FRECUENCIA_CAMPANA_PPD_U6M').distinct().collect()])


['3', '0', '5', '6', '1', '4', '2']


In [9]:
df_base=df_base.withColumnRenamed('LINEA_DIN','LINEA CREDITO DOLARES')
df_base=df_base.withColumnRenamed('CELULAR01','CELULAR1')
df_base=df_base.withColumnRenamed('CELULAR02','CELULAR2')
df_base=df_base.withColumnRenamed('CELULAR03','CELULAR3')
df_base=df_base.withColumnRenamed('CELULAR04','CELULAR4')
df_base=df_base.withColumnRenamed('RECENCIA','RECURRENCIA')
df_base=df_base.withColumnRenamed('FECHA_NACIMIENTO','FEC_NAC')
df_base=df_base.withColumnRenamed('NUEVO_GRUPO03','PROB_CONTACTO')
df_base=df_base.withColumnRenamed('EX_SOCIO','MARCA')
df_base=df_base.withColumnRenamed('FUENTE','N_BASE')

df_base = df_base.withColumn(
    "LIMAPROVINCIA",
    F.when(F.col("PROVINCIA")=='LIMA', F.lit("LIMA"))
    .otherwise(F.lit('PROVINCIA'))
)


In [10]:
df_base=df_base.drop( 'LIMACALLAO')


In [11]:

df_base=df_base.withColumn('AÑO_DURACION_BASE',F.lit('2026'))
df_base=df_base.withColumn('MES_DURACION_BASE',F.lit('07'))
df_base=df_base.withColumnRenamed('TCEA_NEW','TCEA')
df_base=df_base.withColumn('FECHA_ENVIO',F.lit('2026-07-01'))
df_base=df_base.withColumn('SERVICIO',F.lit('02'))



df_base = df_base.withColumn(
    "CEL01",
    F.when(F.col("CELULAR1").rlike(r"^9\d{0,8}$"), F.col("CELULAR1"))
)
df_base = df_base.withColumn(
    "CEL02",
    F.when(F.col("CELULAR2").rlike(r"^9\d{0,8}$"), F.col("CELULAR2"))
)
df_base = df_base.withColumn(
    "CEL03",
    F.when(F.col("CELULAR3").rlike(r"^9\d{0,8}$"), F.col("CELULAR3"))
)
df_base = df_base.withColumn(
    "CEL04",
    F.when(F.col("CELULAR4").rlike(r"^9\d{0,8}$"), F.col("CELULAR4"))
)


In [33]:
df_base.show(2)

+------------+-------+----------------+-----------------+----------------+---------+-------+-------------+---------+--------+--------+--------+-----------+---------+--------+---------------------+--------------------+-----+--------------+----------+---------+-----+------+-----+-----+-----+------+-----+-----+-----+--------+-----+-----+-----+-----+-----+-----+-----+-----+------------------+-----------------+-------------+-----------------+-----------------+-----------+--------+---------+-----+-----+-----+
|ID PROVEEDOR|NOMBRES|APELLIDO PATERNO|TIPO DE DOCUMENTO|NUMERO_DOCUMENTO|TASA(TEA)| TEA_DF|PROB_CONTACTO| CELULAR1|CELULAR2|CELULAR3|CELULAR4|RECURRENCIA|   PERFIL|PRODUCTO|LINEA CREDITO DOLARES|ID RESULTADO GESTION| TCEA|TCEA_CUOTAS_DF|   FEC_NAC|PROVINCIA|L_BCP|L_BBVA|L_IBK|L_SCO|L_BIF|L_CITI|L_FIN|L_RIP|L_CMR|L_CRESCO|L_CEN|L_AZT|L_UNO|L_GNB|L_EFE|L_COM|L_NAC|MARCA|LINEA_ANT_EX_SOCIO|ULT_TASA_EX_SOCIO|LIMAPROVINCIA|AÑO_DURACION_BASE|MES_DURACION_BASE|FECHA_ENVIO|SERVICIO|    CEL01|C

In [ ]:
# df_base=df_base.withColumn('FECHA_ENVIO',F.lit('2026-07-01'))


In [13]:
df_base = (
    df_base
    .withColumn(
        "CELULAR3",
        F.regexp_replace(
            F.col("CELULAR3").cast("string"),
            r"\.0$",
            ""
        )
    )
    .withColumn(
        "CEL03",
        F.when(
            F.col("CELULAR3").rlike(r"^9\d{8}$"),
            F.col("CELULAR3")
        )
    )
)

In [14]:
df_base=df_base.withColumnRenamed('ULT_TASA_EX_SOCIO','TASA_ANT')
df_base=df_base.withColumnRenamed('LINEA_ANT_EX_SOCIO','MAYOR_LINEA')


In [15]:

cols_base = set(df_base.columns)
cols_formato = set(df_formato.columns)

solo_en_base = cols_base - cols_formato
print("Solo en df_base:", solo_en_base)

Solo en df_base: set()


In [16]:
print(df_base.count())
print(df_base.dropDuplicates(['NUMERO_DOCUMENTO']).count())


15547
15547


In [17]:
df_base.groupBy('PROB_CONTACTO') \
    .count() \
    .orderBy('PROB_CONTACTO') \
    .show(30)


+-------------+-----+
|PROB_CONTACTO|count|
+-------------+-----+
|            A|  334|
|            B|  905|
|            C| 3576|
|            D| 5344|
|            E| 4003|
|            F| 1385|
+-------------+-----+



In [44]:
df_base.show(2)

+------------+-------+----------------+-----------------+----------------+---------+-------+-------------+---------+--------+--------+--------+-----------+---------+--------+---------------------+--------------------+-----+--------------+----------+---------+-----+------+-----+-----+-----+------+-----+-----+-----+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----------+--------+-------------+-----------------+-----------------+-----------+--------+---------+-----+-----+-----+
|ID PROVEEDOR|NOMBRES|APELLIDO PATERNO|TIPO DE DOCUMENTO|NUMERO_DOCUMENTO|TASA(TEA)| TEA_DF|PROB_CONTACTO| CELULAR1|CELULAR2|CELULAR3|CELULAR4|RECURRENCIA|   PERFIL|PRODUCTO|LINEA CREDITO DOLARES|ID RESULTADO GESTION| TCEA|TCEA_CUOTAS_DF|   FEC_NAC|PROVINCIA|L_BCP|L_BBVA|L_IBK|L_SCO|L_BIF|L_CITI|L_FIN|L_RIP|L_CMR|L_CRESCO|L_CEN|L_AZT|L_UNO|L_GNB|L_EFE|L_COM|L_NAC|MARCA|MAYOR_LINEA|TASA_ANT|LIMAPROVINCIA|AÑO_DURACION_BASE|MES_DURACION_BASE|FECHA_ENVIO|SERVICIO|    CEL01|CEL02|CEL03|CEL04|
+------------+

In [18]:
append_table_SQL(spark,df_base,'Base_Maestra_Diners_TC',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [20]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_Tc", "SP tNumeros diners_tc")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_Tc", "SP actualizar diners_tc Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_Tc", "SP actualizar diners_tc SA")

SP tNumeros diners_tc | realizado | duración: 200.23 seg
SP actualizar diners_tc Zeus | realizado | duración: 34.32 seg
SP actualizar diners_tc SA | realizado | duración: 5.7 seg


In [ ]:
hugo.lopez@targetoutsourcing.com.pe

In [ ]:
from pyspark.sql import functions as F



In [ ]:
d

Solo en df_base: {'EX_SOCIO', 'NUEVO_GRUPO03', 'LIMACALLAO'}


In [ ]:
df = df_vigente.merge(df_tc, on='NUMERO_DOCUMENTO', how='inner')
list_dni = (
    df['NUMERO_DOCUMENTO']
    .dropna()
    .drop_duplicates()
    .tolist()
)

In [ ]:
df_prueba=df_prueba.drop( 'ULT_TASA_EX_SOCIO', 'LIMACALLAO', 'LINEA_ANT_EX_SOCIO')


In [2]:
df.count()

NUMERO_DOCUMENTO    1833
dtype: int64

In [2]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners_TC
            SET CASHBACK = 'Cash back 300'
            WHERE NUMERO_DOCUMENTO IN ({in_clause})
                and TRY_CONVERT(DATE, fecha_envio) >= TRY_CONVERT(DATE, '{fecha_mes_base}')
                AND TRY_CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, TRY_CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 1771


In [3]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_tc", "SP tNumeros diners TC")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC SA")

SP tNumeros diners TC | realizado | duración: 206.37 seg
SP actualizar diners TC Zeus | realizado | duración: 62.1 seg
SP actualizar diners TC SA | realizado | duración: 2.45 seg


In [ ]:


server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
fecha_mes_base='2026-06-01'
campana=fecha_a_nombre('2026-05-01')
query = f"""
select dni as NumDoc, 1 as venta_target from SAMANTHA.dbo.Ventas_Target
where CAMPANA='Diners'
and CONVERT(DATE, FECHA) >= CONVERT(DATE, '{fecha_mes_base}')
AND CONVERT(DATE, FECHA) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
   
"""
df_venta = pd.read_sql(query, engine_zeus)

df_target = df_tc.merge(
    df_venta,
    on='NumDoc',
    how='left'
)
df_final = df_target.merge(
    df,
    on='NumDoc',
    how='inner'
)
df_final = (
    df_final[df_final['venta_target'].isnull()]
    .drop(columns=['venta_target'])
)
df_final.rename(
    columns={
        'Importe Solicitado': 'Monto'
    },
    inplace=True
)
print(df_final.columns.tolist())

c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


['NumDoc', 'TIPO_PRODUCTO', 'RETIRO', 'Canal', 'Autor', 'Subcanal', 'Motivo', 'Monto', 'fecha', 'hora']


In [72]:
df_final=df_final[df_final['Monto'].notnull()]

In [78]:
df_final[['TIPO_PRODUCTO','Canal', 'Monto', 'fecha']].head()


,TIPO_PRODUCTO,Canal,Monto,fecha
0,PPD,CANALES DIGITALES,41100.0,2026-06-04
1,PPD,CANALES DIGITALES,13800.0,2026-06-04
2,PPD,CONTACT CENTER,6000.0,2026-06-04


In [ ]:

total_monto_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].sum()

total_ope_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].count()

print(f'PPD | Monto total: {int(total_monto_ppd)} |  Total operaciones {int(total_ope_ppd)} ')

PPD | Monto total: 60900 |  Total operaciones 3 


In [ ]:
# ruta_archivo = os.path.join(ruta_csv, 'PPD_plus.xlsx')
# df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
list_dni = (
    df_final.loc[df_final['RETIRO'].isna(), 'NumDoc']
    .dropna()
    .drop_duplicates()
    .tolist()
)

in_clause = ",".join(f"'{x}'" for x in list_dni)

try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners
            SET RETIRO = 'RETIRO'
            WHERE NumDoc IN ({in_clause})
                and CONVERT(DATE, fecha_envio) >= CONVERT(DATE, '{fecha_mes_base}')
                AND CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)


In [2]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 102.27 seg
SP actualizar diners Zeus | realizado | duración: 53.72 seg
SP actualizar diners SA | realizado | duración: 1.35 seg
